# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
bundle exec jekyll build
bundle exec htmlproofer _site --typhoeus-config='{"connecttimeout": 10, "timeout": 30, "max_concurrency": 2}' > htmlproofer-output.txt 2>&1
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [1]:
import pandas as pd

In [59]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 752


In [62]:
df.file.value_counts()

file
_site/es/lecciones/index.html                                                         21
_site/pt/licoes/index.html                                                            20
_site/fr/lecons/index.html                                                            20
_site/fr/index.html                                                                   20
_site/es/lecciones/retirada/index.html                                                19
                                                                                      ..
_site/en/automated-downloading-with-wget/index.html                                    1
_site/en/creating-an-omeka-exhibit/index.html                                          1
_site/en/r-basics-with-tabular-data/index.html                                         1
_site/en/sustainable-authorship-in-plain-text-using-pandoc-and-markdown/index.html     1
_site/en/reviewer-guidelines/index.html                                                1
Name: count, Len

In [61]:
df[(df.message == "'a' tag is missing a reference") ]

,file,line,message


In [64]:
df[df.file == "_site/es/lecciones/index.html"]

,file,line,message
636,_site/es/lecciones/index.html,125,"internally linking to /es/acerca-de, which doe..."
637,_site/es/lecciones/index.html,127,"internally linking to /es/equipo-de-proyecto, ..."
638,_site/es/lecciones/index.html,129,"internally linking to /es/investigacion, which..."
639,_site/es/lecciones/index.html,131,"internally linking to /es/vacantes, which does..."
640,_site/es/lecciones/index.html,133,internally linking to /es/politica-de-privacid...
641,_site/es/lecciones/index.html,144,"internally linking to /es/contribuciones, whic..."
642,_site/es/lecciones/index.html,147,"internally linking to /es/retroalimentacion, w..."
643,_site/es/lecciones/index.html,150,"internally linking to /es/guia-para-revisores,..."
644,_site/es/lecciones/index.html,152,"internally linking to /es/guia-para-autores, w..."
645,_site/es/lecciones/index.html,154,internally linking to /es/guia-para-traductore...


In [9]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/assets/from-html-to-list-of-words-1/obo-...,53,0
1,_site/assets/normaliser-donnees-textuelles-pyt...,53,1
2,_site/en/lessons/sonification.html,27,2
3,_site/en/lessons/collaborative-blog-with-jekyl...,23,3
4,_site/pt/licoes/som-dados-sonificacao-historia...,22,4
...,...,...,...
509,_site/assets/mapping-with-python-leaflet/exerc...,2,509
510,_site/assets/mapping-with-python-leaflet/exerc...,2,510
511,_site/assets/mapping-with-python-leaflet/map/m...,2,511
512,_site/assets/mapping-with-python-leaflet/map/m...,2,512


In [10]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [6]:
merged_df[merged_df.file.str.contains("_site/assets/from-html-to-list-of-words-1/", na=False)].message.value_counts()

message
'a' tag is missing a reference                                                                                              12
internal image i/genericThumb.jpg does not exist                                                                             5
internally linking to static/Contact.jsp, which does not exist                                                               2
internally linking to images.jsp?doc=178006280090, which does not exist                                                      2
internally linking to images.jsp?doc=178006280088, which does not exist                                                      2
internally linking to images.jsp?doc=178006280087, which does not exist                                                      2
internally linking to images.jsp?doc=178006280089, which does not exist                                                      2
internally linking to images.jsp?doc=178006280084, which does not exist                                

In [7]:
merged_df[merged_df.message == "internal image i/genericThumb.jpg does not exist"].file.value_counts()

file
_site/assets/from-html-to-list-of-words-1/obo-t17800628-33.html            5
_site/assets/normaliser-donnees-textuelles-python/obo-t17800628-33.html    5
Name: count, dtype: int64

In [8]:
merged_df[merged_df.file.str.contains("_site/en/lessons/building-static-sites-with-jekyll-github-pages", na=False)]

,file,line,message,count,count_index
412,_site/en/lessons/building-static-sites-with-je...,1334,External link https://support.native-instrumen...,10,88
411,_site/en/lessons/building-static-sites-with-je...,66,External link https://maxcdn.bootstrapcdn.com/...,10,88
410,_site/en/lessons/building-static-sites-with-je...,57,External link https://maxcdn.bootstrapcdn.com/...,10,88
406,_site/en/lessons/building-static-sites-with-je...,117,'a' tag is missing a reference,10,88
413,_site/en/lessons/building-static-sites-with-je...,1474,External link https://jekyllthemes.org/ failed...,10,88
414,_site/en/lessons/building-static-sites-with-je...,1545,External link https://jekyll-windows.juthilo.c...,10,88
415,_site/en/lessons/building-static-sites-with-je...,1548,External link https://chronicle.com/blogs/prof...,10,88
407,_site/en/lessons/building-static-sites-with-je...,136,'a' tag is missing a reference,10,88
408,_site/en/lessons/building-static-sites-with-je...,173,'a' tag is missing a reference,10,88
409,_site/en/lessons/building-static-sites-with-je...,199,'a' tag is missing a reference,10,88


In [30]:
import os
import re

EXTENSIONS = (".yml")

def replace_links_preserving_code_blocks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Match code blocks (triple backticks) and inline code (`...`)
    code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
    modified = content
    offset = 0

    for match in code_blocks:
        start, end = match.span()
        segment = content[start:end]

        # Temporarily mark this section to skip
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified[:start + offset] + placeholder + modified[end + offset:]
        offset += len(placeholder) - (end - start)

    # Replace all http:// with https://
    modified = re.sub(r"http://", "https://", modified)

    # Restore code blocks untouched
    for match in code_blocks:
        start = match.start()
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified.replace(placeholder, match.group(0))

    if content != modified:
        print(f"✅ Updated: {file_path}")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(modified)

def process_all_files(root="."):
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
                replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

process_all_files()

✅ Updated: ./_data/ph_authors.yml
